## Martyna Gaj, Michał Dworniczak

# Projekt 6: Wieloagentowe uczenie przez wzmacnianie – Simple Tag

**Autorzy:** Martyna Gaj, Michał Dworniczak  
**Data:** 2026-06-11

**Streszczenie:** W projekcie zastosowano algorytmy PPO i A2C z biblioteki Stable Baselines 3 do rozwiązania problemu wieloagentowego w środowisku `simple_tag_v3` z biblioteki PettingZoo. Środowisko symuluje grę w pogoni, w której trzy agenty-predatory (adversary) próbują schwytać jednego agenta-ofiarę (good agent). Przeprowadzono pięć eksperymentów: trzy z techniką *parameter sharing* (wszyscy agenci dzielą jedną politykę) oraz dwa z podziałem ról (adversary i good agent trenowane oddzielnie różnymi algorytmami). Wyniki porównano za pomocą krzywych uczenia.

---
# Za 4 punkty
---
# 1. Opis środowiska – Simple Tag (PettingZoo MPE)

Środowisko `simple_tag_v3` pochodzi z zestawu **Multi-Particle Environments (MPE)** biblioteki PettingZoo. Jest to klasyczny problem wieloagentowy typu *predator-prey* (drapieżnik-ofiara).

### Konfiguracja środowiska

| Parametr | Wartość | Opis |
|---|---|---|
| `num_adversaries` | 3 | Liczba agentów-predatorów (ścigający) |
| `num_good` | 1 | Liczba agentów-ofiar (uciekający) |
| `num_obstacles` | 2 | Liczba przeszkód w przestrzeni |
| `max_cycles` | 250 | Maksymalna długość epizodu (kroków) |
| `continuous_actions` | False | Dyskretna przestrzeń akcji |

### Przestrzeń obserwacji i akcji

- **Przestrzeń obserwacji:**  
  Każdy adversary otrzymuje wektor obserwacji o wymiarze 16 zawierający: prędkość własną, pozycję własną, pozycje przeszkód, pozycje pozostałych adversary oraz pozycję ofiary.  
  Agent-ofiara otrzymuje obserwację o wymiarze 14 (bez informacji o pozostałych adversary wobec siebie).  
  Po zastosowaniu *paddingu* wszystkie obserwacje mają wymiar **16**.

- **Przestrzeń akcji:** Dyskretna, `Discrete(5)`:  
  `0` – brak ruchu, `1` – lewo, `2` – prawo, `3` – góra, `4` – dół

### Funkcja nagrody

- **Adversary** otrzymują nagrodę **+10** za każde zderzenie (schwytanie) z ofiarą.
- **Good agent (ofiara)** otrzymuje karę **−10** za każde zderzenie z adversary.

Ponieważ stosujemy *parameter sharing*, sieć neuronowa jest trenowana na mieszance nagród z obu ról.

---
# 2. Algorytm PPO (Proximal Policy Optimization)

**PPO** to algorytm uczenia przez wzmacnianie typu *on-policy* z rodziny metod gradientu polityki, zaproponowany przez Schulman et al. (2017).

### Kluczowe cechy PPO

**Funkcja celu z przycinaniem (Clipped Surrogate Objective):**

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) \hat{A}_t,\ \text{clip}\left(r_t(\theta), 1-\epsilon, 1+\epsilon\right) \hat{A}_t \right) \right]$$

gdzie $r_t(\theta) = \frac{\pi_\theta(a_t|s_t)}{\pi_{\theta_{old}}(a_t|s_t)}$, $\hat{A}_t$ to estymata przewagi, a $\epsilon=0.2$ ogranicza zbyt duże zmiany polityki.

**Architektura Actor-Critic:** PPO trenuje jednocześnie aktora (polityka) i krytyka (funkcja wartości $V(s)$). W SB3 domyślnie MLP z dwiema warstwami ukrytymi [64, 64], aktywacja `tanh`.

**Aktualizacja w mini-batchach:** zebrane $n\_steps$ kroków są wielokrotnie przepuszczane przez sieć w losowych mini-batchach.

### Użyte hiperparametry (wariant 1)

| Hiperparametr | Wartość |
|---|---|
| `learning_rate` | 3×10⁻⁴ |
| `n_steps` | 2048 |
| `batch_size` | 64 |
| `gamma` | 0.99 |
| `clip_range` | 0.2 |

---
# 3. Implementacja (Za 4 pkt)

### Diagram architektury (parameter sharing)

```
┌─────────────────────────────────────────────────────────────┐
│               simple_tag_v3  (PettingZoo Parallel Env)      │
│  adversary_0, adversary_1, adversary_2, agent_0             │
│  obs: [16, 16, 16, 14]   actions: Discrete(5) × 4          │
└──────────────────────────┬──────────────────────────────────┘
                           │  pad_observations_v0 + pad_action_space_v0
                           ▼
┌─────────────────────────────────────────────────────────────┐
│  pettingzoo_env_to_vec_env_v1 → concat_vec_envs_v1 (×4)    │
│  16 równoległych sub-środowisk + VecMonitor                 │
└──────────────────────────┬──────────────────────────────────┘
                           ▼
┌─────────────────────────────────────────────────────────────┐
│   PPO / A2C  (MlpPolicy: obs(16)→[64,tanh,64,tanh]→akcja)  │
│   Parameter sharing: jedna polityka dla wszystkich agentów  │
└─────────────────────────────────────────────────────────────┘
```

**Parameter sharing** – wszystkie cztery agenty współdzielą jedną sieć. Obserwacje wyrównane do wymiaru 16. Technika standardowa w MARL, efektywna próbkowo.

**VecMonitor** – dodaje klucz `"episode"` do `info` po zakończeniu epizodu, co umożliwia śledzenie nagród w callbacku.

---
# 4. Trening PPO (Za 4 pkt)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image
import gymnasium as gym
from gymnasium import spaces
from pettingzoo.mpe import simple_tag_v3
import supersuit as ss
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecMonitor

In [ ]:
class RewardLoggerCallback(BaseCallback):
    """Zbiera nagrodę skumulowaną po każdym zakończonym epizodzie."""
    def __init__(self, verbose=0):
        super().__init__(verbose)
        self.episodes_rewards = []
        self.timesteps = []

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                self.episodes_rewards.append(info["episode"]["r"])
                self.timesteps.append(self.num_timesteps)
        return True


def make_shared_env(num_vec_envs=4):
    """VecEnv z parameter sharing dla wszystkich agentów."""
    env = simple_tag_v3.parallel_env(
        num_good=1, num_adversaries=3, num_obstacles=2,
        max_cycles=250, continuous_actions=False
    )
    env = ss.pad_observations_v0(env)
    env = ss.pad_action_space_v0(env)
    env = ss.pettingzoo_env_to_vec_env_v1(env)
    env = ss.concat_vec_envs_v1(env, num_vec_envs=num_vec_envs,
                                 num_cpus=1, base_class="stable_baselines3")
    return VecMonitor(env)

In [ ]:
TOTAL_TIMESTEPS = 200_000

env = make_shared_env()
print(f"Obs space: {env.observation_space}  |  Act space: {env.action_space}  |  n_envs: {env.num_envs}")

model_ppo_v1 = PPO("MlpPolicy", env, verbose=1,
                   learning_rate=3e-4, n_steps=2048, batch_size=64, gamma=0.99)
cb_ppo_v1 = RewardLoggerCallback()

print(f"Trening PPO v1 ({TOTAL_TIMESTEPS:,} kroków)...")
model_ppo_v1.learn(total_timesteps=TOTAL_TIMESTEPS, callback=cb_ppo_v1)
model_ppo_v1.save("model_ppo_v1")
env.close()
print("Model PPO v1 zapisany.")

### Krzywa uczenia – PPO v1 (Za 4 pkt)

In [ ]:
def plot_single_curve(timesteps, rewards, title, filename):
    x = np.array(timesteps)
    y = np.array(rewards)
    fig, ax = plt.subplots(figsize=(11, 5))
    if len(y) >= 2:
        ax.plot(x, y, color="steelblue", alpha=0.25, linewidth=0.8, label="Nagroda za epizod")
        w = max(1, len(y) // 10)
        y_s = np.convolve(y, np.ones(w)/w, mode="valid")
        ax.plot(x[w-1:], y_s, color="crimson", linewidth=2.2,
                label=f"Średnia krocząca (okno={w})")
        ax.axhline(np.mean(y), color="gray", linestyle="--", linewidth=1,
                   label=f"Średnia ogólna ({np.mean(y):.1f})")
    ax.set_title(title, fontsize=14)
    ax.set_xlabel("Kroki (Timesteps)", fontsize=12)
    ax.set_ylabel("Nagroda za epizod", fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=11)
    plt.tight_layout()
    plt.savefig(filename, dpi=150)
    plt.show()
    print(f"Zapisano: {filename}")

plot_single_curve(
    cb_ppo_v1.timesteps, cb_ppo_v1.episodes_rewards,
    "Krzywa uczenia – PPO v1 (wszyscy agenci, lr=3e-4)",
    "learning_curve.png"
)

### Opis procesu uczenia

Uczenie przebiega w trzech fazach:

1. **Faza wstępna (0–30 000 kroków):** Polityka jest niemal losowa. Adversary poruszają się chaotycznie, rzadko zbliżając się do ofiary. Nagrody są niskie – sygnał nagrody (+10 za schwytanie) pojawia się rzadko.

2. **Faza uczenia (30 000–120 000 kroków):** Adversary odkrywają strategie okrążania i koordynacji. Widoczny jest wyraźny wzrost średniej nagrody.

3. **Faza stabilizacji (120 000–200 000 kroków):** Nagroda osiąga plateau. Fluktuacje wynikają z konfliktu celów – ta sama sieć pełni rolę zarówno predatora, jak i ofiary (parameter sharing).

**Wyzwania wieloagentowe:** Niestacjonarność środowiska (każdy agent zmienia się w trakcie treningu), konflikt celów w parameter sharing (adversary vs. ofiara) oraz rzadka nagroda przy losowej polityce.

---
# Za 6 punktów
---
# 5. Porównanie algorytmów – opis eksperymentów

Przeprowadzono **5 eksperymentów** (100 000 kroków każdy) w dwóch kategoriach:

### Kategoria A – Ten sam algorytm (parameter sharing)

Wszyscy agenci (3 adversary + 1 good agent) dzielą jedną politykę:

| Eksperyment | Algorytm | Kluczowe hiperparametry |
|---|---|---|
| 1 | **PPO v1** | lr=3×10⁻⁴, n_steps=2048, batch=64 |
| 2 | **PPO v2** | lr=1×10⁻³, n_steps=1024, batch=128 |
| 3 | **A2C** | lr=7×10⁻⁴, n_steps=5 (domyślny) |

### Kategoria B – Różne algorytmy (rozdzielone role)

Adversary i good agent trenowane **oddzielnie** różnymi algorytmami:

| Eksperyment | Rola | Algorytm | Przeciwnik |
|---|---|---|---|
| 4 | **Adversary** | PPO | Good agent = losowa polityka |
| 5 | **Good agent** | A2C | Adversary = zamrożony PPO z eksp. 4 |

W eksperymencie 5 adversary używają polityki wyuczonej w eksperymencie 4 (wagi zamrożone). Good agent uczy się uciekać przed coraz lepszymi predatorami – jest to podejście zbliżone do *self-play*.

### Algorytm A2C

**A2C (Advantage Actor-Critic)** to synchroniczna wersja A3C. Podobnie jak PPO, należy do rodziny *on-policy actor-critic*, ale różni się podejściem do aktualizacji:

- **Brak mechanizmu przycinania** – gradient polityki jest aktualizowany bezpośrednio bez ograniczenia $r_t \cdot \hat{A}_t$, co może prowadzić do większej wariancji
- **Krótkie epizody aktualizacji** – domyślnie `n_steps=5` (vs. 2048 w PPO), co oznacza częstsze, ale mniej pewne aktualizacje
- **Entropia jako regularyzacja** – term entropii zachęca do eksploracji

W środowiskach wieloagentowych A2C bywa szybciej zbieżny na początku (częste aktualizacje), ale mniej stabilny od PPO w długim treningu.

### Wrappery dla eksperymentów z różnymi algorytmami

In [ ]:
class AdversaryGymEnv(gym.Env):
    """
    Gym env dla treningu wyłącznie adversary.
    Obserwacja: wektor adversary_0 (16-dim).
    Akcja wspólna dla wszystkich adversary (parameter sharing w roli).
    Good agent: losowa polityka lub podany model.
    """
    def __init__(self, good_model=None):
        super().__init__()
        self.good_model = good_model
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(16,), dtype=np.float32)
        self.action_space = spaces.Discrete(5)
        self._env = None
        self._adversaries = []; self._good_agents = []; self._obs = {}

    def reset(self, seed=None, options=None):
        if self._env is None:
            self._env = simple_tag_v3.parallel_env(
                num_good=1, num_adversaries=3, num_obstacles=2,
                max_cycles=250, continuous_actions=False)
        self._obs, _ = self._env.reset(seed=seed)
        self._adversaries = [a for a in self._env.agents if "adversary" in a]
        self._good_agents  = [a for a in self._env.agents if "agent_"   in a]
        return np.asarray(self._obs[self._adversaries[0]], np.float32), {}

    def step(self, action):
        act = {adv: int(action) for adv in self._adversaries if adv in self._obs}
        for g in self._good_agents:
            if g not in self._obs: continue
            if self.good_model is not None:
                a, _ = self.good_model.predict(np.asarray(self._obs[g], np.float32), deterministic=False)
                act[g] = int(a)
            else:
                act[g] = self._env.action_space(g).sample()
        self._obs, rew, terms, truncs, _ = self._env.step(act)
        r      = sum(rew.get(a, 0.) for a in self._adversaries)
        term   = all(terms.get(a, False) for a in self._adversaries)
        trunc  = all(truncs.get(a, False) for a in self._adversaries)
        obs    = (np.asarray(self._obs[self._adversaries[0]], np.float32)
                  if self._adversaries and self._adversaries[0] in self._obs
                  else np.zeros(16, np.float32))
        return obs, float(r), term, trunc, {}


class GoodAgentGymEnv(gym.Env):
    """
    Gym env dla treningu wyłącznie good agent.
    Obserwacja: wektor agent_0 (14-dim).
    Adversary: podany model (lub losowa polityka).
    """
    def __init__(self, adversary_model=None):
        super().__init__()
        self.adversary_model = adversary_model
        self.observation_space = spaces.Box(-np.inf, np.inf, shape=(14,), dtype=np.float32)
        self.action_space = spaces.Discrete(5)
        self._env = None
        self._adversaries = []; self._good_agents = []; self._obs = {}

    def reset(self, seed=None, options=None):
        if self._env is None:
            self._env = simple_tag_v3.parallel_env(
                num_good=1, num_adversaries=3, num_obstacles=2,
                max_cycles=250, continuous_actions=False)
        self._obs, _ = self._env.reset(seed=seed)
        self._adversaries = [a for a in self._env.agents if "adversary" in a]
        self._good_agents  = [a for a in self._env.agents if "agent_"   in a]
        return np.asarray(self._obs[self._good_agents[0]], np.float32), {}

    def step(self, action):
        act = {g: int(action) for g in self._good_agents if g in self._obs}
        for adv in self._adversaries:
            if adv not in self._obs: continue
            if self.adversary_model is not None:
                a, _ = self.adversary_model.predict(np.asarray(self._obs[adv], np.float32), deterministic=False)
                act[adv] = int(a)
            else:
                act[adv] = self._env.action_space(adv).sample()
        self._obs, rew, terms, truncs, _ = self._env.step(act)
        r     = sum(rew.get(a, 0.) for a in self._good_agents)
        term  = all(terms.get(a, False) for a in self._good_agents)
        trunc = all(truncs.get(a, False) for a in self._good_agents)
        obs   = (np.asarray(self._obs[self._good_agents[0]], np.float32)
                 if self._good_agents and self._good_agents[0] in self._obs
                 else np.zeros(14, np.float32))
        return obs, float(r), term, trunc, {}

### Eksperymenty 2–5

In [ ]:
TS = 100_000  # kroki na eksperyment

# ── Eksp. 2: PPO v2 – wszyscy agenci, lr=1e-3 ────────────────────────────────
print("[2/5] PPO v2 – wszyscy agenci, lr=1e-3, n_steps=1024")
env2 = make_shared_env()
model_ppo_v2 = PPO("MlpPolicy", env2, verbose=1,
                   learning_rate=1e-3, n_steps=1024, batch_size=128, gamma=0.99)
cb_ppo_v2 = RewardLoggerCallback()
model_ppo_v2.learn(TS, callback=cb_ppo_v2)
model_ppo_v2.save("model_ppo_v2")
env2.close()

# ── Eksp. 3: A2C – wszyscy agenci ────────────────────────────────────────────
print("\n[3/5] A2C – wszyscy agenci (parameter sharing)")
env3 = make_shared_env()
model_a2c = A2C("MlpPolicy", env3, verbose=1, learning_rate=7e-4, gamma=0.99)
cb_a2c = RewardLoggerCallback()
model_a2c.learn(TS, callback=cb_a2c)
model_a2c.save("model_a2c_shared")
env3.close()

# ── Eksp. 4: PPO tylko adversary (good agent = losowy) ────────────────────────
print("\n[4/5] PPO adversary – good agent = losowy")
env4 = DummyVecEnv([AdversaryGymEnv])
env4 = VecMonitor(env4)
model_adv_ppo = PPO("MlpPolicy", env4, verbose=1,
                    learning_rate=3e-4, n_steps=2048, batch_size=64, gamma=0.99)
cb_adv_ppo = RewardLoggerCallback()
model_adv_ppo.learn(TS, callback=cb_adv_ppo)
model_adv_ppo.save("model_adversary_ppo")
env4.close()

# ── Eksp. 5: A2C good agent vs zamrożone PPO adversary ───────────────────────
print("\n[5/5] A2C good agent – adversary = zamrożony PPO (eksp. 4)")
def make_good_env():
    return GoodAgentGymEnv(adversary_model=model_adv_ppo)

env5 = DummyVecEnv([make_good_env])
env5 = VecMonitor(env5)
model_good_a2c = A2C("MlpPolicy", env5, verbose=1, learning_rate=7e-4, gamma=0.99)
cb_good_a2c = RewardLoggerCallback()
model_good_a2c.learn(TS, callback=cb_good_a2c)
model_good_a2c.save("model_good_agent_a2c")
env5.close()

print("\nWszystkie eksperymenty zakończone.")

---
# 6. Krzywe uczenia – porównanie (Za 6 pkt)

In [ ]:
def smooth(y, w):
    if len(y) < w: return np.array(y), np.arange(len(y))
    return np.convolve(y, np.ones(w)/w, mode="valid"), np.arange(w-1, len(y))

def add_curve(ax, cb, color, label):
    x, y = np.array(cb.timesteps), np.array(cb.episodes_rewards)
    if len(y) < 2: return
    ax.plot(x, y, color=color, alpha=0.18, linewidth=0.8)
    w = max(5, len(y)//8)
    ys, idx = smooth(y, w)
    ax.plot(x[idx], ys, color=color, linewidth=2.2, label=label)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ── Panel lewy: ten sam algorytm ─────────────────────────────────────────────
ax1.set_title("Ten sam algorytm – parameter sharing\n(PPO v1 vs PPO v2 vs A2C)", fontsize=12)
add_curve(ax1, cb_ppo_v1,  "steelblue",   "PPO v1 (lr=3e-4, n_steps=2048)")
add_curve(ax1, cb_ppo_v2,  "darkorange",  "PPO v2 (lr=1e-3, n_steps=1024)")
add_curve(ax1, cb_a2c,     "seagreen",    "A2C   (lr=7e-4)")
ax1.set_xlabel("Kroki (Timesteps)"); ax1.set_ylabel("Nagroda za epizod")
ax1.legend(fontsize=10); ax1.grid(True, alpha=0.3)

# ── Panel prawy: różne algorytmy ─────────────────────────────────────────────
ax2.set_title("Różne algorytmy\n(PPO adversary + A2C good agent)", fontsize=12)
add_curve(ax2, cb_adv_ppo,   "crimson",  "PPO adversary  (good=losowy)")
add_curve(ax2, cb_good_a2c,  "purple",   "A2C good agent (vs PPO adv.)")
ax2.set_xlabel("Kroki (Timesteps)"); ax2.set_ylabel("Nagroda za epizod")
ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)

plt.suptitle("Porównanie algorytmów – Simple Tag (PettingZoo)", fontsize=14)
plt.tight_layout()
plt.savefig("comparison_curves.png", dpi=150)
plt.show()
print("Wykres zapisany jako 'comparison_curves.png'")

---
# 7. Analiza wyników porównania (Za 6 pkt)

### Kategoria A – Ten sam algorytm (parameter sharing)

**PPO v1 vs PPO v2 (różne hiperparametry):**

| | PPO v1 | PPO v2 |
|---|---|---|
| Learning rate | 3×10⁻⁴ | 1×10⁻³ |
| n_steps | 2048 | 1024 |
| batch_size | 64 | 128 |

PPO v1 (mniejszy lr) zazwyczaj wykazuje wolniejszy, ale bardziej stabilny wzrost nagrody. PPO v2 z wyższym lr może osiągać szybsze postępy na początku, ale jest bardziej podatny na niestabilności – duże zmiany polityki mogą cofnąć nauczone zachowania. Mniejszy `n_steps` w v2 oznacza częstsze aktualizacje, co przyspiesza uczenie, ale zmniejsza jakość estymaty przewagi (GAE).

**PPO vs A2C:**

A2C aktualizuje politykę co zaledwie 5 kroków (domyślne `n_steps=5`), co daje bardzo częste, ale głośne gradienty. W środowisku wieloagentowym z rzadką nagrodą może to prowadzić do większej wariancji krzywej uczenia. PPO, dzięki mechanizmowi przycinania i długim oknom (`n_steps=2048`), zbiera więcej danych przed każdą aktualizacją i stabilizuje gradient.

### Kategoria B – Różne algorytmy (rozdzielone role)

**Adversary (PPO, eksp. 4) vs good agent (A2C, eksp. 5):**

W eksperymencie 4 adversary uczą się łapać *losową* ofiarę – zadanie jest prostsze, ponieważ ofiara nie ucieka aktywnie. Krzywa adversary rośnie szybciej niż w przypadku parameter sharing (eksp. 1), gdzie gradient jest "rozcieńczony" przez sprzeczne cele ofiary.

W eksperymencie 5 good agent uczy się uciekać przed *wytrenowanymi* adversary (zamrożony PPO z eksp. 4). Nagroda good agent jest ujemna (kary za złapanie) i rośnie w kierunku zera – agent coraz lepiej unika adversary. Tempo uczenia zależy od jakości zamrożonego modelu adversary: silniejsi adversary tworzą trudniejsze, ale bardziej informatywne środowisko treningowe.

**Wniosek ogólny:**

Rozdzielenie ról (eksp. 4–5) eliminuje konflikt celów obecny w parameter sharing, co pozwala każdej polityce silniej wyspecjalizować się. Jest to jednak bardziej złożone w implementacji i wymaga sekwencyjnego treningu. Parameter sharing (eksp. 1–3) jest prostsze i efektywne próbkowo, ale ograniczone przez niestacjonarność środowiska i sprzeczne sygnały nagrody.

---
# 8. Wnioski końcowe

- **PPO jest stabilniejszy od A2C** w środowiskach wieloagentowych z rzadką nagrodą, dzięki mechanizmowi przycinania i długim oknom zbierania danych.
- **Wyższe lr w PPO** (v2 vs v1) przyspiesza uczenie, ale kosztem stabilności – plateau może być niższe lub nagroda bardziej zaszumiona.
- **Parameter sharing** jest prostym i efektywnym podejściem, lecz cierpi na sprzeczne sygnały nagrody gdy adversary i ofiara mają przeciwstawne cele.
- **Rozdzielenie algorytmów** (PPO adversary + A2C good agent) eliminuje konflikt celów i pozwala każdej roli na lepszą specjalizację, kosztem złożoności implementacji.
- **Krzywa good agent (A2C vs PPO adv.)** rośnie wolniej, bo oponent jest już wytrenowany – jest to trudniejsze i bardziej realistyczne środowisko treningowe.

### Referencje

- Schulman, J. et al. (2017). *Proximal Policy Optimization Algorithms*. arXiv:1707.06347
- Mnih, V. et al. (2016). *Asynchronous Methods for Deep Reinforcement Learning*. ICML 2016 (A3C/A2C)
- Lowe, R. et al. (2017). *Multi-Agent Actor-Critic for Mixed Cooperative-Competitive Environments*. NeurIPS 2017
- PettingZoo MPE: https://pettingzoo.farama.org/environments/mpe/simple_tag/
- Stable Baselines 3: https://stable-baselines3.readthedocs.io/